## Setup

This notebook lives next to `archive (3)/`. Put your API key in a local `.env`
file (`ANTHROPIC_API_KEY=...`). Shared agent code is in `agent_core.py`; the
Streamlit UI is `app.py`.

Swap the `MODEL` string in `agent_core.py` (or pass `model=` to the helpers) to
change provider.


In [ ]:
from pathlib import Path
import sys

# Resolve project root whether cwd is this folder or a parent
CWD = Path.cwd().resolve()
if (CWD / "archive (3)").exists():
    PROJECT_DIR = CWD
elif (CWD / "Agentic_data_explorer" / "archive (3)").exists():
    PROJECT_DIR = CWD / "Agentic_data_explorer"
else:
    PROJECT_DIR = Path(".").resolve()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from agent_core import (
    DATA_DIR,
    MODEL,
    VIZ_TOOLS,
    ask_explorer,
    ask_viz,
    extract_charts,
    get_explorer,
    get_viz_agent,
    redact_text,
    reset_viz_agent,
    safe_read_csv,
)
from langchain.messages import HumanMessage

reset_viz_agent()  # pick up newly added tools after code edits
explorer = get_explorer()
viz_agent = get_viz_agent()

print("Model:", MODEL)
print("Project dir:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Files:", [p.name for p in sorted(DATA_DIR.glob("*.csv"))])
print("Viz tools:", [t.name for t in VIZ_TOOLS])
print("Explorer + viz agents ready (PII-safe).")


## Explore the dataset

The explorer agent writes and runs PII-safe Python (`safe_read_csv`, no `user_id`).


In [ ]:
TASK = """Explore the dataset in archive (3)/ from scratch using Python you write
and run yourself — PII-safe only (no user_id, no customer-level stats). Cover:
1. What files exist and how big they are
2. Columns and a few sample rows per file (use safe_read_csv)
3. Row counts (exact for small files, estimated or sampled for huge ones)
4. One interesting cross-file insight (e.g. products linked to departments)
5. A short summary of what this dataset is about"""

result = explorer.invoke({"messages": [HumanMessage(TASK)]})
print(redact_text(result["messages"][-1].content))


In [ ]:
print("--- agent trace: every tool call the agent made ---\n")
for m in result["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:500])
    print()


## Ask your own question

Change the message below and re-run — the agent will write fresh Python and run it.


In [ ]:
follow_up = explorer.invoke({"messages": [
    HumanMessage("Which department has the most products? Show the code you ran."),
]})
print(redact_text(follow_up["messages"][-1].content))


## Visualization agent (10 presentation tools)

The viz agent aggregates with `run_python` / `filter_dataframe`, then uses:

1. `recommend_chart` → best chart type  
2. `create_visualization` → chart spec for the frontend  
3. `ask_visualization_critic` → second-agent review  
4. `generate_insights` / `explain_visualization`  
5. `generate_dashboard_layout` → `build_dashboard`  
6. `change_theme` / `export_report` when asked  

Use `ask_viz(...)` (recommended) or invoke `viz_agent` directly. Restart Streamlit after editing `agent_core.py`.


In [ ]:
# Example: line chart — orders by hour of day
viz_out = ask_viz("Show a line chart of orders by hour of day")
print(viz_out["answer"])
print("\nCharts returned:", len(viz_out["charts"]))
for chart in viz_out["charts"]:
    print(chart)


In [ ]:
# Render charts inside the notebook with Plotly
from charts import chart_to_figure

for chart in viz_out["charts"]:
    fig = chart_to_figure(chart)
    fig.show()


In [ ]:
# More chart types
bar_out = ask_viz("Bar chart of top 10 departments by product count")
print(bar_out["answer"])
for chart in bar_out["charts"]:
    chart_to_figure(chart).show()


## Frontend

From this folder:

```bash
pip install -r requirements.txt
streamlit run app.py
```
